In [1]:
!pip install datasets

In [2]:
from datasets import load_dataset

In [3]:
ds=load_dataset('arunapb/nutrition5k-foodseg103')

In [4]:
print(ds)

DatasetDict({
    train: Dataset({
        features: ['image', 'id', 'split', 'ingredients', 'fs103_classes', 'total_calories', 'total_mass', 'total_fat', 'total_carb', 'total_protein'],
        num_rows: 891
    })
    test: Dataset({
        features: ['image', 'id', 'split', 'ingredients', 'fs103_classes', 'total_calories', 'total_mass', 'total_fat', 'total_carb', 'total_protein'],
        num_rows: 100
    })
})


In [5]:
print(ds['train'].column_names)

['image', 'id', 'split', 'ingredients', 'fs103_classes', 'total_calories', 'total_mass', 'total_fat', 'total_carb', 'total_protein']


In [6]:
print(ds['train'][0])

{'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=640x640 at 0x2237699DFD0>, 'id': 'dish_1567629081', 'split': 'train', 'ingredients': [{'calories': 239.31, 'carb': 30.15, 'fat': 8.46, 'grams': 90.0, 'id': 'ingr_0000000011', 'name': 'cheese pizza', 'protein': 10.08}], 'fs103_classes': ['pizza'], 'total_calories': 239.309998, 'total_mass': 90.0, 'total_fat': 8.46, 'total_carb': 30.150002, 'total_protein': 10.08}


In [7]:
print(ds['train'][0]['image'])
print(ds['train'][0]['fs103_classes'])
print(ds['train'][0]['total_calories'])

<PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=640x640 at 0x22376A54410>
['pizza']
239.309998


In [8]:
!pip install datasets transformers torch torchvision scikit_learn pandas pillow matplotlib

In [9]:
import os
import numpy as np
import pandas as pd 
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import Dataset,DataLoader
from torchvision import transforms,models
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_absolute_error

In [10]:
# df=ds['train'].to_pandas()
# print(df.head())

In [11]:
# #فقط ستون های مورد نیاز
# df=df[['image','fs103_classes','total_calories']].copy()

In [12]:
#حذف داده های ناقص
#df=df.dropna(subset=['image','fs103_classes','total_calories'])

In [13]:
#df['food_name']=df['fs103_classes'].apply(lambda x:x[0] if isinstance(x,list) else str(x))

In [14]:
#کالری را عدد میکنیم
#df['total_calories']=pd.to_numeric(df['total_calories'],errors='coerce')

In [15]:
#حذف کالری های نامعتبر
#df=df.dropna(subset=['total_calories']).reset_index(drop=True)

In [16]:
#print(df.head())

In [17]:
#food_names=sorted(df['food_name'].unique())

In [18]:
#food_to_id={name:i for i,name in enumerate(food_names)}

In [19]:
#id_to_food={i:name for name,i in food_to_id.items()}

In [20]:
#df['food_id']=df['food_name'].map(food_to_id)

In [21]:
#NUM_CLASSES=len(food_names)

In [22]:
# print('Number of classes:',NUM_CLASSES)
# print(food_to_id)

In [23]:
# print('Number of samples:',len(ds['train']))

In [24]:
# print('Number fo food classes:',len(set(x[0] for x in ds['train']['fs103_classes'])))

In [25]:
# print('First image:',ds['train'][0]['image'])

In [26]:
# print('Image type:',type(ds['train'][0]['image']))

In [27]:
print(ds)
print(ds['train'][0])

DatasetDict({
    train: Dataset({
        features: ['image', 'id', 'split', 'ingredients', 'fs103_classes', 'total_calories', 'total_mass', 'total_fat', 'total_carb', 'total_protein'],
        num_rows: 891
    })
    test: Dataset({
        features: ['image', 'id', 'split', 'ingredients', 'fs103_classes', 'total_calories', 'total_mass', 'total_fat', 'total_carb', 'total_protein'],
        num_rows: 100
    })
})
{'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=640x640 at 0x22371111810>, 'id': 'dish_1567629081', 'split': 'train', 'ingredients': [{'calories': 239.31, 'carb': 30.15, 'fat': 8.46, 'grams': 90.0, 'id': 'ingr_0000000011', 'name': 'cheese pizza', 'protein': 10.08}], 'fs103_classes': ['pizza'], 'total_calories': 239.309998, 'total_mass': 90.0, 'total_fat': 8.46, 'total_carb': 30.150002, 'total_protein': 10.08}


In [28]:
#تعیین کلاس های غذا
train_data=ds['train']
test_data=ds['test']
print(train_data)
print(test_data)

Dataset({
    features: ['image', 'id', 'split', 'ingredients', 'fs103_classes', 'total_calories', 'total_mass', 'total_fat', 'total_carb', 'total_protein'],
    num_rows: 891
})
Dataset({
    features: ['image', 'id', 'split', 'ingredients', 'fs103_classes', 'total_calories', 'total_mass', 'total_fat', 'total_carb', 'total_protein'],
    num_rows: 100
})


In [29]:
#استخراج نام کلاس ها
all_classes=[]
for item in train_data:
    cls=item['fs103_classes']
    #اگرکلاس به صورت لیست باشد
    if isinstance(cls,list):
        all_classes.extend(cls)
    else:
        all_classes.append(cls)
classes=sorted(list(set(all_classes)))
print('Number of classes:',len(classes))
print(classes[:20])

Number of classes: 42
['almond', 'apple', 'asparagus', 'avocado', 'banana', 'bread', 'broccoli', 'carrot', 'cauliflower', 'celery stick', 'cheese butter', 'cherry', 'chicken duck', 'corn', 'cucumber', 'dried cranberries', 'egg', 'eggplant', 'enoki mushroom', 'fish']


In [30]:
#ساخت مپینگ
class_to_idx={cls_name: idx for idx, cls_name in enumerate(classes)}
idx_to_class={idx:cls_name for cls_name,idx in class_to_idx.items()}
print(class_to_idx)

{'almond': 0, 'apple': 1, 'asparagus': 2, 'avocado': 3, 'banana': 4, 'bread': 5, 'broccoli': 6, 'carrot': 7, 'cauliflower': 8, 'celery stick': 9, 'cheese butter': 10, 'cherry': 11, 'chicken duck': 12, 'corn': 13, 'cucumber': 14, 'dried cranberries': 15, 'egg': 16, 'eggplant': 17, 'enoki mushroom': 18, 'fish': 19, 'grape': 20, 'green beans': 21, 'ice cream': 22, 'lettuce': 23, 'melon': 24, 'olives': 25, 'onion': 26, 'orange': 27, 'pasta': 28, 'pear': 29, 'pepper': 30, 'pineapple': 31, 'pizza': 32, 'potato': 33, 'rice': 34, 'salad': 35, 'sausage': 36, 'steak': 37, 'tofu': 38, 'tomato': 39, 'watermelon': 40, 'wonton dumplings': 41}


In [31]:
#ترنسفورم عکس ها
train_trainsform=transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2,contrast=0.2,saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],std=[0.229,0.224,0.225])
])
test_transform=transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],std=[0.229,0.224,0.225])
])

In [32]:
#ساخت دیتاست
class FoodDataset(Dataset):
    def __init__(self,hf_dataset,transform=None):
        self.dataset=hf_dataset
        self.transform=transform
    def __len__(self):
        return len(self.dataset)
    def __getitem__(self,idx):
        item=self.dataset[idx]
        image=item['image']
        if image.mode !='RGB':
            image=image.convert('RGB')
        if self.transform:
            image=self.transform(image)
        food_class=item['fs103_classes']
        if isinstance(food_class,list):
            food_class=food_class[0]
        label=class_to_idx[food_class]
        calories=float(item['total_calories'])
        return{
            'image':image,
            'label':torch.tensor(label,dtype=torch.long),
            'calories':torch.tensor(calories,dtype=torch.float32)
        }

In [33]:
#ساخت دیتالودر
train_dataset=FoodDataset(train_data,transform=train_trainsform)
test_dataset=FoodDataset(test_data,transform=test_transform)

In [34]:
train_loader=DataLoader(train_dataset,batch_size=32,shuffle=True,num_workers=0)
test_loader=DataLoader(test_dataset,batch_size=32,shuffle=False,num_workers=0)

In [35]:
#بررسی یک نمونه
sample=train_dataset[0]
print(sample['image'].shape)
print(sample['label'])
print(idx_to_class[sample['label'].item()])
print(sample['calories'])

torch.Size([3, 224, 224])
tensor(32)
pizza
tensor(239.3100)


In [36]:
#ساخت مدل
device= torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:',device)

Device: cpu


In [37]:
#EfficientNet

In [38]:
weights=models.EfficientNet_B0_Weights.DEFAULT
model=models.efficientnet_b0(weights=weights)
num_features = model.classifier[1].in_features
model.classifier = nn.Identity()

In [39]:
class FoodNutritionModel(nn.Module):
    def __init__(self, backbone, num_features, num_classes):
        super().__init__()
        self.backbone = backbone
        # تشخیص غذا
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(num_features, num_classes)
        )
        # تخمین کالری
        self.calorie_regressor = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(num_features, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )
    def forward(self, x):
        features = self.backbone(x)
        class_output = self.classifier(features)
        calorie_output = self.calorie_regressor(features)
        return class_output, calorie_output


In [40]:
#مدل را بساز:
model = FoodNutritionModel(
    backbone=model,
    num_features=num_features,
    num_classes=len(classes)
)

model = model.to(device)

In [41]:
criterion_class = nn.CrossEntropyLoss()
criterion_calorie = nn.MSELoss()

In [42]:
#Optimizer:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

In [43]:
!pip install tqdm

In [44]:
from tqdm.auto import tqdm

In [ ]:
#آموزش مدل
num_epochs = 15
for epoch in range(num_epochs):
    model.train()
    running_loss = 0
    correct = 0
    total = 0
    progress = tqdm(
        train_loader,
        desc=f"Epoch {epoch+1}/{num_epochs}"
    )
    for batch in progress:
        images = batch["image"].to(device)
        labels = batch["label"].to(device)
        calories = batch["calories"].to(device)
        optimizer.zero_grad()
        class_output, calorie_output = model(images)
        calorie_output = calorie_output.squeeze(1)
        loss_class = criterion_class(
            class_output,
            labels
        )
        loss_calorie = criterion_calorie(
            calorie_output,
            calories
        )
        # ترکیب دو loss
        loss = loss_class + 0.001 * loss_calorie
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        predictions = torch.argmax(
            class_output,
            dim=1
        )
        correct += (
            predictions == labels
        ).sum().item()
        total += labels.size(0)
        progress.set_postfix({
            "loss": running_loss / (progress.n + 1),
            "acc": correct / total
        })
    epoch_acc = correct / total
    print(
        f"\nEpoch {epoch+1}: "
        f"Loss={running_loss/len(train_loader):.4f}, "
        f"Accuracy={epoch_acc:.4f}"
    )

Epoch 1/15:   0%|          | 0/28 [00:00<?, ?it/s]


Epoch 1: Loss=50.5597, Accuracy=0.1762


Epoch 2/15:   0%|          | 0/28 [00:00<?, ?it/s]


Epoch 2: Loss=48.8847, Accuracy=0.4512


Epoch 3/15:   0%|          | 0/28 [00:00<?, ?it/s]

...

Epoch 10/15:   0%|          | 0/28 [00:00<?, ?it/s]


Epoch 10: Loss=14.4029, Accuracy=0.2368


Epoch 11/15:   0%|          | 0/28 [00:00<?, ?it/s]


Epoch 11: Loss=12.9363, Accuracy=0.2536


Epoch 12/15:   0%|          | 0/28 [00:00<?, ?it/s]

In [ ]:
#ذخیره مدل
torch.save({
    "model_state_dict": model.state_dict(),
    "classes": classes
}, "food_calorie_model.pth")


In [ ]:
#پیش‌بینی از روی عکس
def predict_food(image_path):
    model.eval()
    image = Image.open(image_path).convert("RGB")
    image_tensor = test_transform(image)
    image_tensor = image_tensor.unsqueeze(0)
    image_tensor = image_tensor.to(device)
    with torch.no_grad():
        class_output, calorie_output = model(
            image_tensor
        )
    # کلاس
    probabilities = torch.softmax(
        class_output,
        dim=1
    )

    confidence, predicted_class = torch.max(
        probabilities,
        dim=1
    )

    food_name = idx_to_class[
        predicted_class.item()
    ]
    # کالری
    calories = calorie_output.item()
    # کالری منفی معنی ندارد
    calories = max(0, calories)

    return {"food": food_name,
        "calories": calories,
        "confidence": confidence.item()
    }

In [ ]:
#تست
result = predict_food(r"C:\Users\P\Desktop\paria\Deep\PIZZA.jpg")

print("Food:", result["food"])
print("Calories:", result["calories"])
print("Confidence:", result["confidence"])

In [ ]:
result = predict_food(r"C:\Users\P\Desktop\paria\Deep\3.jpg")

print("Food:", result["food"])
print("Calories:", result["calories"])
print("Confidence:", result["confidence"])